# HSEQ 3 · Gold
Owned by: Quality and HSE manager. Reads from other domains only through shortcuts.

In [ ]:
# Domain: hseq · owner: Quality and HSE manager
# Attach this notebook to the lakehouse "HSEQ" in the workspace "Solvi HSEQ".
DOMAIN = "hseq"
LANDING_PATH = f"Files/landing/{DOMAIN}"   # upload this domain's CSV files here
TABLE_FORMAT = "delta"
LOCAL_PREFIX = ""                            # local testing sets this to the domain name
AS_OF = "2026-10-31"                         # demo data ends here; use the current date with live data

from pyspark.sql import functions as F, Window as W

def write_table(df, name):
    (df.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable(LOCAL_PREFIX + name))
    print(f"wrote {name:<30} {df.count():>9,} rows")

def read_table(name):                         # a table this domain owns
    return spark.read.table(LOCAL_PREFIX + name)

def read_shared(domain, name):                # another domain's table, through a OneLake shortcut named domain + name
    return spark.read.table(f"{domain}{name}")

## Compliance: is every installation audit-ready?

In [ ]:
inst = read_shared("assets", "diminstallation")
write_table(read_table("silvercompliancerequirement"), "dimcompliancerequirement")
ci = (read_table("silvercomplianceitem").withColumn("status",
        F.when(F.col("submitted_date").isNotNull() & (F.col("submitted_date") <= F.col("due_date")), "On time")
         .when(F.col("submitted_date").isNotNull(), "Late").when(F.col("due_date") < F.lit(AS_OF), "Overdue").otherwise("Not yet due")))
write_table(ci, "factcomplianceitem")
dev = (read_table("silverinspectiondeviation").withColumn("is_open", F.col("closed_date").isNull())
        .withColumn("days_open", F.datediff(F.coalesce("closed_date", F.lit(AS_OF).cast("date")), "found_date")))
write_table(dev, "factinspectiondeviation")
overdue = ci.filter("status = 'Overdue'").groupBy("installation_id").agg(F.count("*").alias("overdue_items"))
open_high = dev.filter("is_open AND severity = 'High'").groupBy("installation_id").agg(F.count("*").alias("open_high_deviations"))
audit = (inst.select("installation_id").join(overdue, "installation_id", "left").join(open_high, "installation_id", "left")
         .fillna(0, ["overdue_items","open_high_deviations"])
         .withColumn("audit_ready", (F.col("overdue_items") == 0) & (F.col("open_high_deviations") == 0)))
write_table(audit, "factauditreadiness")

## Battery risk: act on conditions, not on counts
Counting days since the last incident tells nobody what to do on Tuesday. Instead every battery job is checked for the conditions that
make incidents likely, across the whole chain: supplier, transport, storage, crew, site and planning. The factors, weights and
stop rules live in `risk_rule`, owned by quality, so they can be changed without touching code.

- **Red**: a stop factor is present, or the score is 6 or more. Fix it before work starts.
- **Yellow**: score 3 to 5. Supervisor check and toolbox talk before start.
- **Green**: proceed.

In [ ]:
rules = read_table("silverriskrule")
job = read_shared("delivery", "factjob")
dl = read_shared("supply", "factbatterydelivery")
crew = read_shared("delivery", "dimcrew")
orders = read_shared("sales", "factorderline").select("order_id","customer_id").distinct()
site = read_shared("core", "dimcustomer").select("customer_id","site_id","municipality","country")
stations = read_shared("core", "dimsite").select("site_id","station_id")
fc = read_shared("core", "factweatherforecast").select(F.col("station_id").alias("fc_station"), F.col("date").alias("fc_date"), "temp_min_c")

PREQUAL = read_shared("supply", "factsupplierprequal")   # subcontractor safety prequalification windows, owned by purchasing

j = (job.join(dl.select("delivery_id","supplier_id","packaging_damage","shock_indicator_tripped","inspected_at_receipt","temp_excursion","storage_days"), "delivery_id")
        .join(crew.select("crew_id", F.col("supplier_id").alias("subcontractor_id"), "crew_type"), "crew_id")
        .join(orders, "order_id").join(site, "customer_id").join(stations, "site_id")
        .join(fc, (F.col("station_id") == F.col("fc_station")) & (F.col("job_date") == F.col("fc_date")), "left").drop("fc_station", "fc_date"))
pq = (j.select("job_id","subcontractor_id","job_date").join(PREQUAL, (F.col("subcontractor_id") == PREQUAL.supplier_id) &
        F.col("job_date").between(F.col("valid_from"), F.col("valid_to")), "left")
        .groupBy("job_id").agg((F.count("valid_from") > 0).alias("prequal_ok")))
j = j.join(pq, "job_id")
done = F.col("status") == "Done"
indoor = F.col("location") != "Outdoor cabinet"
flags = {   # factor_code -> condition. Planned jobs use what is known now: delivery, crew, site and the forecast.
  "DAMAGE_UNCHECKED": ((F.col("packaging_damage") == 1) | (F.col("shock_indicator_tripped") == 1)) & (F.col("inspected_at_receipt") == 0),
  "PKG_DAMAGE": F.col("packaging_damage") == 1, "SHOCK": F.col("shock_indicator_tripped") == 1,
  "NOT_INSPECTED": F.col("inspected_at_receipt") == 0, "TEMP_EXCURSION": F.col("temp_excursion") == 1,
  "LONG_STORAGE": F.col("storage_days") > 60,
  "PREQUAL_EXPIRED": F.col("subcontractor_id").isNotNull() & ~F.col("prequal_ok"),
  "UNCERTIFIED": ~F.col("crew_certified"), "NEW_CREW": F.col("crew_prior_jobs") < 3,
  "COLD_BATTERY": done & (F.col("battery_temp_c") < 5) & (F.col("acclimatised") == 0),
  "FROST_FORECAST": ~done & (F.col("temp_min_c") < 0),
  "NO_VENT_CHECK": done & indoor & (F.col("ventilation_check_done") == 0),
  "GARAGE": F.col("location") == "Garage",
  "NO_SJA": F.when(done, F.col("sja_done") == 0).otherwise(F.lit(False)),
  "LATE": F.when(done, F.col("days_late") > 14).otherwise(F.lit(False)),
}
present = None
for code, cond in flags.items():
    part = j.filter(F.coalesce(cond, F.lit(False))).select("job_id", F.lit(code).alias("factor_code"))
    present = part if present is None else present.unionByName(part)
fact_job_risk_factor = present.join(rules, "factor_code")
write_table(fact_job_risk_factor, "factjobriskfactor")

score = fact_job_risk_factor.groupBy("job_id").agg(F.sum("weight").alias("risk_score"), F.max("stops_work").alias("has_stop_factor"),
                                                   F.concat_ws(", ", F.sort_array(F.collect_list("factor_code"))).alias("factors"))
fact_job_risk = (j.select("job_id","job_date","status","order_id","customer_id","municipality","country","crew_id","crew_type","supplier_id","subcontractor_id","location")
  .join(score, "job_id", "left").fillna({"risk_score": 0, "has_stop_factor": 0, "factors": ""})
  .withColumn("risk_level", F.when((F.col("has_stop_factor") == 1) | (F.col("risk_score") >= 6), "Red")
                             .when(F.col("risk_score") >= 3, "Yellow").otherwise("Green")))
write_table(fact_job_risk, "factjobrisk")
write_table(read_table("silverhseevent"), "facthseevent")
write_table(read_table("silverhseaction"), "facthseaction")

## Proof: do the factors predict events?
Events per 100 jobs by risk level, before and after the rule was introduced on 1 April 2026.

In [ ]:
ev = read_table("facthseevent").filter("event_type <> 'Good catch'").groupBy("job_id").agg(F.count("*").alias("events"))
r = read_table("factjobrisk").filter("status = 'Done'").join(ev, "job_id", "left").fillna({"events": 0})
display(r.withColumn("period", F.when(F.col("job_date") >= "2026-04-01", "After rule").otherwise("Before rule"))
         .groupBy("period", "risk_level").agg(F.count("*").alias("jobs"), F.round(F.sum("events") / F.count("*") * 100, 1).alias("events_per_100_jobs"))
         .orderBy("period", "risk_level"))